# Generating text from gpt 2

In [1]:
import copy

import transformers

In [2]:
import numpy as np
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer, set_seed

In [3]:
# Load model and tokenizer
model = GPT2LMHeadModel.from_pretrained("gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [6]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [7]:
import spacy

## Decoding from GPT2

This tutorial investigates how to use GPT2 (the forerunner of GPT3) to generate text.  There are a number of ways to do this that trade-off the realism of the text against the amount of variation.

At every stage, GPT2 takes an input string and returns a probability for each of the possible subsequent tokens.  We can choose what to do with these probability.  We could always *greedily choose* the most likely next token, or we could draw a *sample* randomly according to the probabilities.  There are also intermediate strategies such as *top-k sampling* and *nucleus sampling*, that have some controlled randomness.

We'll also investigate *beam search* -- the idea is that rather than greedily take the next best token at each stage, we maintain a set of hypotheses  (beams)as we add each subsequent token and return the most likely overall hypothesis.  This is not necessarily the same result we get from greedily choosing the next token.

In [8]:
np.random.seed(42)

In [9]:
tokenizer.vocab_size

50257

## 50 tokens random

In [10]:
for i in range(2000, 2050):
    # index = np.random.randint(tokenizer.vocab_size)
    print(
        "Token: %d " % i + tokenizer.decode(torch.tensor(i), skip_special_tokens=True)
    )

Token: 2000  mind
Token: 2001 aff
Token: 2002 omm
Token: 2003  future
Token: 2004 ged
Token: 2005  cut
Token: 2006  tot
Token: 2007 itch
Token: 2008  video
Token: 2009  investig
Token: 2010  net
Token: 2011  My
Token: 2012 rict
Token: 2013 ien
Token: 2014 .)
Token: 2015  impro
Token: 2016 though
Token: 2017 wards
Token: 2018  connect
Token: 2019  Med
Token: 2020 selves
Token: 2021 ensive
Token: 2022 mb
Token: 2023 ober
Token: 2024 ators
Token: 2025 An
Token: 2026  50
Token: 2027  redu
Token: 2028 resent
Token: 2029  above
Token: 2030  fre
Token: 2031  Europe
Token: 2032 sw
Token: 2033  amount
Token: 2034  App
Token: 2035  either
Token: 2036  milit
Token: 2037  anal
Token: 2038  fail
Token: 2039  En
Token: 2040 ales
Token: 2041  special
Token: 2042  black
Token: 2043 IT
Token: 2044 cher
Token: 2045  looking
Token: 2046  fire
Token: 2047 yn
Token: 2048  almost
Token: 2049 oon


## Sampling

Each time we run GPT2 it will take in a set of tokens, and return a probability over each of the possible next tokens.  The simplest thing we could do is to just draw a sample from this probability distribution each time.

In [11]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [14]:
# Put this inside your training loop after loss.backward() but before optimizer.step()
for name, param in model.named_parameters():
    print(f"{name=} {param.shape=},{param.requires_grad}")

name='transformer.wte.weight' param.shape=torch.Size([50257, 768]),True
name='transformer.wpe.weight' param.shape=torch.Size([1024, 768]),True
name='transformer.h.0.ln_1.weight' param.shape=torch.Size([768]),True
name='transformer.h.0.ln_1.bias' param.shape=torch.Size([768]),True
name='transformer.h.0.attn.c_attn.weight' param.shape=torch.Size([768, 2304]),True
name='transformer.h.0.attn.c_attn.bias' param.shape=torch.Size([2304]),True
name='transformer.h.0.attn.c_proj.weight' param.shape=torch.Size([768, 768]),True
name='transformer.h.0.attn.c_proj.bias' param.shape=torch.Size([768]),True
name='transformer.h.0.ln_2.weight' param.shape=torch.Size([768]),True
name='transformer.h.0.ln_2.bias' param.shape=torch.Size([768]),True
name='transformer.h.0.mlp.c_fc.weight' param.shape=torch.Size([768, 3072]),True
name='transformer.h.0.mlp.c_fc.bias' param.shape=torch.Size([3072]),True
name='transformer.h.0.mlp.c_proj.weight' param.shape=torch.Size([3072, 768]),True
name='transformer.h.0.mlp.c_pr

In [19]:
list(model.named_modules())

[('',
  GPT2LMHeadModel(
    (transformer): GPT2Model(
      (wte): Embedding(50257, 768)
      (wpe): Embedding(1024, 768)
      (drop): Dropout(p=0.1, inplace=False)
      (h): ModuleList(
        (0-11): 12 x GPT2Block(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
          (attn): GPT2Attention(
            (c_attn): Conv1D(nf=2304, nx=768)
            (c_proj): Conv1D(nf=768, nx=768)
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (resid_dropout): Dropout(p=0.1, inplace=False)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): GPT2MLP(
            (c_fc): Conv1D(nf=3072, nx=768)
            (c_proj): Conv1D(nf=768, nx=3072)
            (act): NewGELUActivation()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
    )
    (lm_head): Linear(in_features=7

In [20]:
model.get_submodule("lm_head").requires_grad = False

In [21]:
set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors="pt")
outputs = model(
    input_ids=input_tokens["input_ids"],
    attention_mask=input_tokens["attention_mask"],
)

In [22]:
# Embeddings
outputs.logits, outputs.logits.detach().shape

(tensor([[[ -36.2871,  -35.0111,  -38.0791,  ...,  -40.5161,  -41.3757,
            -34.9190],
          [ -95.2851,  -94.9462, -101.8512,  ...,  -99.5187, -101.3784,
            -96.4179],
          [ -84.2826,  -83.2167,  -90.9228,  ...,  -93.4169,  -95.1053,
            -85.6533],
          [ -81.9151,  -82.0943,  -85.0690,  ...,  -84.9429,  -87.7480,
            -81.9450],
          [ -67.8890,  -68.5929,  -72.5499,  ...,  -76.7069,  -76.1264,
            -70.6933],
          [ -80.7412,  -81.7387,  -87.3581,  ...,  -89.1840,  -89.0606,
            -84.2771]]], grad_fn=<UnsafeViewBackward0>),
 torch.Size([1, 6, 50257]))

In [23]:
dir(outputs)

['__annotate_func__',
 '__annotations_cache__',
 '__class__',
 '__class_getitem__',
 '__contains__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__ior__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__or__',
 '__post_init__',
 '__reduce__',
 '__reduce_ex__',
 '__replace__',
 '__repr__',
 '__reversed__',
 '__ror__',
 '__setattr__',
 '__setitem__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 'attentions',
 'clear',
 'copy',
 'cross_attentions',
 'fromkeys',
 'get',
 'hidden_states',
 'items',
 'keys',
 'logits',
 'loss',
 'move_to_end',
 'past_key_values',
 'pop',
 'popitem',
 'setdefault',
 'to_tuple',
 'update',
 'values']

In [24]:
print(list(outputs.keys()))

['logits', 'past_key_values']


In [25]:
embs = torch.ones((32, 100, 128))
B, T, _ = embs.shape
pos_ids = torch.arange(T).expand(B, -1)
print(f"{pos_ids.shape=}")
pos_E = torch.nn.Embedding(200, 128)
print(pos_E)
x = pos_E(pos_ids)

pos_ids.shape=torch.Size([32, 100])
Embedding(200, 128)


In [26]:
dir(x)

['H',
 'T',
 '__abs__',
 '__add__',
 '__and__',
 '__annotate_func__',
 '__array__',
 '__array_priority__',
 '__array_wrap__',
 '__bool__',
 '__class__',
 '__complex__',
 '__contains__',
 '__deepcopy__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__div__',
 '__dlpack__',
 '__dlpack_c_exchange_api__',
 '__dlpack_device__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__float__',
 '__floordiv__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__iand__',
 '__idiv__',
 '__ifloordiv__',
 '__ilshift__',
 '__imod__',
 '__imul__',
 '__index__',
 '__init__',
 '__init_subclass__',
 '__int__',
 '__invert__',
 '__ior__',
 '__ipow__',
 '__irshift__',
 '__isub__',
 '__iter__',
 '__itruediv__',
 '__ixor__',
 '__le__',
 '__len__',
 '__long__',
 '__lshift__',
 '__lt__',
 '__matmul__',
 '__mod__',
 '__module__',
 '__mul__',
 '__ne__',
 '__neg__',
 '__new__',
 '__nonzero__',
 '__or__',
 '__pos__',
 '__pow__',
 '__radd_

In [27]:
prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0, -1]

In [28]:
prob_over_tokens

array([7.3242154e-05, 2.7012617e-05, 9.7970556e-08, ..., 1.5779333e-08,
       1.7852384e-08, 2.1336391e-06], shape=(50257,), dtype=float32)

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Your original tensor setup
embs = torch.ones((32, 100, 128))
B, T, d_model = embs.shape
pos_ids = torch.arange(T).expand(B, -1)
pos_E = nn.Embedding(200, 128)
x = pos_E(pos_ids)  # Shape: [32, 100, 128]

# 2. Define your vocabulary size (e.g., 1000 tokens in your dictionary)
vocab_size = 1000

# 3. Create a language model head (un-embedding layer)
# This projects from your hidden dimension (128) to your token scores (1000)
lm_head = nn.Linear(d_model, vocab_size, bias=False)

# 4. Step 1: Calculate raw scores (Logits)
logits = lm_head(x)  # Shape becomes: [32, 100, 1000]

# 5. Step 2: Convert logits into a valid probability distribution
# We apply softmax over dim=-1 (the 1000 vocabulary choices)
token_probabilities = F.softmax(logits, dim=-1)

print(f"{token_probabilities.shape=}")
# Output: token_probabilities.shape=torch.Size([32, 100, 1000])

# Verify that the probabilities for each token position sum up to exactly 1.0
print("Sum of probabilities at first token:", token_probabilities[0, 0].sum().item())
# Output: 1.0

token_probabilities.shape=torch.Size([32, 100, 1000])
Sum of probabilities at first token: 0.9999999403953552


In [30]:
probability_over_tokens = token_probabilities.detach().numpy()[0, -1]

In [31]:
probability_over_tokens

array([0.00199308, 0.00188893, 0.00072851, 0.00066889, 0.00024005,
       0.00044199, 0.00071142, 0.00025538, 0.00061564, 0.00101604,
       0.00108428, 0.00128831, 0.00072905, 0.00041271, 0.00061425,
       0.00063   , 0.00060915, 0.00045306, 0.00078176, 0.00056498,
       0.00061403, 0.00073574, 0.00074168, 0.00104787, 0.00015421,
       0.00087529, 0.00219612, 0.00062138, 0.00027855, 0.00017207,
       0.00084269, 0.00133663, 0.00383795, 0.00056754, 0.00327548,
       0.00137203, 0.00041418, 0.00059434, 0.00077821, 0.00159816,
       0.00146977, 0.00144302, 0.00142776, 0.00082617, 0.00067638,
       0.00135541, 0.00082058, 0.00209242, 0.00080252, 0.0007887 ,
       0.00083265, 0.00039136, 0.00066873, 0.0009351 , 0.00292504,
       0.00146487, 0.00072124, 0.00059633, 0.00059094, 0.00077424,
       0.0009654 , 0.00121197, 0.00100383, 0.00067107, 0.00054677,
       0.00305002, 0.00050186, 0.00050243, 0.00051694, 0.00110377,
       0.00112206, 0.00089741, 0.001445  , 0.00107053, 0.00223

In [32]:
prob_over_tokens[:10]

array([7.3242154e-05, 2.7012617e-05, 9.7970556e-08, 7.1807058e-08,
       5.5681386e-07, 7.1281420e-08, 6.4150308e-06, 3.4432414e-06,
       1.5285304e-05, 2.0660384e-06], dtype=float32)

In [33]:
prob_over_tokens.sum()

np.float32(1.0000323)

In [34]:
np.random.choice(len(prob_over_tokens), 1, p=prob_over_tokens)

array([326])

In [35]:
def sample_next_token(input_tokens, model, tokenizer):
    # Run model to get prediction over next output
    outputs = model(
        input_ids=input_tokens["input_ids"],
        attention_mask=input_tokens["attention_mask"],
    )
    # Find prediction
    prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0, -1]
    # TODO Draw a random token according to the probabilities
    # next_token should be an array with an sole integer in it (as below)
    # Use:  https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html
    # Replace this line
    next_token = np.random.choice(len(prob_over_tokens), 1, p=prob_over_tokens)

    # Append token to sentence
    output_tokens = input_tokens
    output_tokens["input_ids"] = torch.cat(
        (output_tokens["input_ids"], torch.tensor([next_token])), dim=1
    )
    output_tokens["attention_mask"] = torch.cat(
        (output_tokens["attention_mask"], torch.tensor([[1]])), dim=1
    )
    output_tokens["last_token_prob"] = prob_over_tokens[next_token]

    return output_tokens

In [36]:
np.random.choice(len(prob_over_tokens), 1, p=prob_over_tokens)

array([345])

In [37]:
# TODO Modify the code below by changing the number of tokens generated and the initial sentence
# to get a feel for how well this works.  Since I didn't reset the seed, it will give a different
# answer every time that you run it.

# TODO Experiment with changing this line:
set_seed(0)
input_txt = "How fast is this gpt?"
input_tokens = tokenizer(input_txt, return_tensors="pt")
# TODO Experiment with changing this line:
for i in range(20):
    input_tokens = sample_next_token(input_tokens, model, tokenizer)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

/var/folders/px/m0g9wbyn1sv678fsx9lgsm600000gn/T/ipykernel_3190/727354976.py:19: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:255.)
  (output_tokens["input_ids"], torch.tensor([next_token])), dim=1


How fast is this gpt? That
How fast is this gpt? That said
How fast is this gpt? That said,
How fast is this gpt? That said, by
How fast is this gpt? That said, by using
How fast is this gpt? That said, by using take
How fast is this gpt? That said, by using takeout
How fast is this gpt? That said, by using takeout formula
How fast is this gpt? That said, by using takeout formula formula
How fast is this gpt? That said, by using takeout formula formula I
How fast is this gpt? That said, by using takeout formula formula I mean
How fast is this gpt? That said, by using takeout formula formula I mean it
How fast is this gpt? That said, by using takeout formula formula I mean it would
How fast is this gpt? That said, by using takeout formula formula I mean it would definitely
How fast is this gpt? That said, by using takeout formula formula I mean it would definitely be
How fast is this gpt? That said, by using takeout formula formula I mean it would definitely be the
How fast is this gpt?

## Greedy token selection

You probably (correctly) got the impression that the text from pure sampling of the probability model can be kind of random.  How about if we choose most likely token at each step?


In [38]:
np.argmax(prob_over_tokens)

np.int64(326)

In [39]:
def get_best_next_token(input_tokens, model, tokenizer):
    # Run model to get prediction over next output
    outputs = model(
        input_ids=input_tokens["input_ids"],
        attention_mask=input_tokens["attention_mask"],
    )
    # Find prediction
    prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0, -1]

    next_token = [np.argmax(prob_over_tokens)]

    # Append token to sentence
    output_tokens = input_tokens
    output_tokens["input_ids"] = torch.cat(
        (output_tokens["input_ids"], torch.tensor([next_token])), dim=1
    )
    output_tokens["attention_mask"] = torch.cat(
        (output_tokens["attention_mask"], torch.tensor([[1]])), dim=1
    )
    output_tokens["last_token_prob"] = prob_over_tokens[next_token]
    return output_tokens

In [40]:
# TODO Modify the code below by changing the number of tokens generated and the initial sentence
# to get a feel for how well this works.  Since I didn't reset the seed, it will give a different
# answer every time that you run it.

# TODO Experiment with changing this line:
set_seed(0)
input_txt = "How fast is this gpt?"
input_tokens = tokenizer(input_txt, return_tensors="pt")
# TODO Experiment with changing this line:
for i in range(20):
    input_tokens = get_best_next_token(input_tokens, model, tokenizer)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

How fast is this gpt?

How fast is this gpt?


How fast is this gpt?

I
How fast is this gpt?

I'm
How fast is this gpt?

I'm not
How fast is this gpt?

I'm not sure
How fast is this gpt?

I'm not sure.
How fast is this gpt?

I'm not sure. I
How fast is this gpt?

I'm not sure. I'm
How fast is this gpt?

I'm not sure. I'm not
How fast is this gpt?

I'm not sure. I'm not sure
How fast is this gpt?

I'm not sure. I'm not sure if
How fast is this gpt?

I'm not sure. I'm not sure if it
How fast is this gpt?

I'm not sure. I'm not sure if it's
How fast is this gpt?

I'm not sure. I'm not sure if it's a
How fast is this gpt?

I'm not sure. I'm not sure if it's a good
How fast is this gpt?

I'm not sure. I'm not sure if it's a good idea
How fast is this gpt?

I'm not sure. I'm not sure if it's a good idea to
How fast is this gpt?

I'm not sure. I'm not sure if it's a good idea to use
How fast is this gpt?

I'm not sure. I'm not sure if it's a good idea to use a


In [41]:
import pandas as pd

df = pd.DataFrame.from_dict(
    {"word": tokenizer.vocab.keys(), "frequency": tokenizer.vocab.values()}
)

In [42]:
df.to_csv("gpt2_vocab.csv")

## Top-K sampling

You probably noticed that the greedy strategy produces quite realistic text, but it's kind of boring.  It produces generic answers.  Also, if this was a chatbot, then we wouldn't necessarily want it to produce the same answer to a question each time.

Top-K sampling is a compromise strategy that samples randomly from the top K most probable tokens.  We could just choose them with a uniform distribution, or (as here) we could sample them according to their original probabilities.

In [43]:
def get_top_k_tokens(input_tokens, model, tokenizer, k=20):
    # Run model to get prediction over next output
    outputs = model(
        input_ids=input_tokens["input_ids"],
        attention_mask=input_tokens["attention_mask"],
    )
    # Find prediction
    prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0, -1]

    # Draw a sample from the top K most likely tokens.
    # Take copy of the probabilities and sort from largest to smallest (use np.sort)
    # TODO -- replace this line
    sorted_prob_over_tokens = np.sort(prob_over_tokens)[::-1]

    # Find the probability at the k'th position
    # TODO -- replace this line
    kth_prob_value = sorted_prob_over_tokens[k]

    # Set all probabilities below this value to zero
    prob_over_tokens[prob_over_tokens < kth_prob_value] = 0

    # Renormalize the probabilities so that they sum to one

    prob_over_tokens = prob_over_tokens / prob_over_tokens.sum()

    # Draw random token
    next_token = np.random.choice(
        len(prob_over_tokens), 1, replace=False, p=prob_over_tokens
    )[0]

    # Append token to sentence
    output_tokens = input_tokens
    output_tokens["input_ids"] = torch.cat(
        (output_tokens["input_ids"], torch.tensor([[next_token]])), dim=1
    )
    output_tokens["attention_mask"] = torch.cat(
        (output_tokens["attention_mask"], torch.tensor([[1]])), dim=1
    )
    output_tokens["last_token_prob"] = prob_over_tokens[next_token]
    return output_tokens

In [44]:
# Expected output:
# The best thing about Bath is that you get to see all the beautiful faces of

set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors="pt")
for i in range(10):
    input_tokens = get_top_k_tokens(input_tokens, model, tokenizer, k=10)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

The best thing about Bath is that
The best thing about Bath is that you
The best thing about Bath is that you get
The best thing about Bath is that you get to
The best thing about Bath is that you get to see
The best thing about Bath is that you get to see all
The best thing about Bath is that you get to see all the
The best thing about Bath is that you get to see all the beautiful
The best thing about Bath is that you get to see all the beautiful faces
The best thing about Bath is that you get to see all the beautiful faces of


## Nucleus sampling

Top-K sampling has the disadvantage that sometimes there are only a few plausible next tokens, and sometimes there are a lot.  How do we adapt to this situation?  One way is to sample from a fixed proportion of the probability mass.  That is we order the tokens in terms of probability and cut off the possibility of sampling when the cumulative sum is greater than a threshold.

This way, we adapt the number of possible tokens that we can choose.

In [45]:
def get_nucleus_sampling_token(input_tokens, model, tokenizer, thresh=0.2):
    outputs = model(
        input_ids=input_tokens["input_ids"],
        attention_mask=input_tokens["attention_mask"],
    )
    # Find prediction
    prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0, -1]
    sorted_probability = np.sort(prob_over_tokens)
    threshold = sorted_probability[thresh]
    prob_over_tokens = prob_over_tokens[prob_over_tokens < threshold]

In [46]:
def get_nucleus_sampling_token(input_tokens, model, tokenizer, thresh=0.25):
    # Run model to get prediction over next output
    outputs = model(
        input_ids=input_tokens["input_ids"],
        attention_mask=input_tokens["attention_mask"],
    )
    # Find prediction
    prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0, -1]

    # Find the most likely tokens that make up the first (thresh) of the probability
    sorted_probs_decreasing = prob_over_tokens[np.argsort(prob_over_tokens)[::-1]]

    cum_sum_probs = np.cumsum(sorted_probs_decreasing)

    thresh_index = int(np.argmax(cum_sum_probs >= thresh)) + 1
    print("Choosing from %d tokens" % (thresh_index))
    # probability value to threshold

    thresh_prob = prob_over_tokens[thresh_index]

    # Set any probabilities less than this to zero
    prob_over_tokens[prob_over_tokens < thresh_prob] = 0
    # Renormalize
    prob_over_tokens = prob_over_tokens / np.sum(prob_over_tokens)
    # Draw random token
    next_token = np.random.choice(
        len(prob_over_tokens), 1, replace=False, p=prob_over_tokens
    )

    # Append token to sentence
    output_tokens = input_tokens
    output_tokens["input_ids"] = torch.cat(
        (output_tokens["input_ids"], torch.tensor([next_token])), dim=1
    )
    output_tokens["attention_mask"] = torch.cat(
        (output_tokens["attention_mask"], torch.tensor([[1]])), dim=1
    )
    output_tokens["last_token_prob"] = prob_over_tokens[next_token]
    return output_tokens

In [1]:
from transformers import BertModel

In [47]:
# prob_over_tokens[prob_over_tokens < sorted_probs_decreasing[0]] = 0

In [49]:
import numpy as np
import torch
import torch.nn.functional as F


def get_nucleus_sampling_token(input_tokens, model, tokenizer, thresh=0.25):
    # Run model to get prediction over next output
    outputs = model(
        input_ids=input_tokens["input_ids"],
        attention_mask=input_tokens["attention_mask"],
    )

    # Extract probabilities for the last token position
    prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().cpu().numpy()[0, -1]

    # 1. Sort indices and probabilities in descending order
    sorted_indices = np.argsort(prob_over_tokens)[::-1]
    sorted_probs_decreasing = prob_over_tokens[sorted_indices]

    # 2. Cumulative sum to find Top-P cutoff
    cum_sum_probs = np.cumsum(sorted_probs_decreasing)

    # Find the first index where cumulative sum exceeds the threshold
    # Add 1 to ensure we include the token that crosses the threshold
    cutoff_index = int(np.argmax(cum_sum_probs >= thresh)) + 1
    print(f"Choosing from {cutoff_index} tokens")

    # Get the tokens and their values that fall within our nucleus
    nucleus_indices = sorted_indices[:cutoff_index]

    # Create a fresh mask array so we don't accidentally mutate state incorrectly
    filtered_probs = np.zeros_like(prob_over_tokens)
    filtered_probs[nucleus_indices] = prob_over_tokens[nucleus_indices]

    # 3. Renormalize the remaining probabilities
    filtered_probs = filtered_probs / np.sum(filtered_probs)

    # Draw random token based on filtered distribution
    next_token_arr = np.random.choice(
        len(filtered_probs), 1, replace=False, p=filtered_probs
    )
    next_token_id = int(next_token_arr[0])

    # Append token to sentence securely
    output_tokens = (
        input_tokens.copy()
    )  # Avoid mutating original dict in place if reused

    # Concatenate tracking correct tensor shape dimensions
    output_tokens["input_ids"] = torch.cat(
        (
            output_tokens["input_ids"],
            torch.tensor([[next_token_id]], device=output_tokens["input_ids"].device),
        ),
        dim=1,
    )
    output_tokens["attention_mask"] = torch.cat(
        (
            output_tokens["attention_mask"],
            torch.tensor([[1]], device=output_tokens["attention_mask"].device),
        ),
        dim=1,
    )
    output_tokens["last_token_prob"] = prob_over_tokens[next_token_id]

    return output_tokens

In [50]:
import torch
import torch.nn.functional as F


def get_nucleus_sampling_token_torch(input_tokens, model, tokenizer, thresh=0.25):
    with torch.no_grad():
        # Run model to get prediction over next output
        outputs = model(
            input_ids=input_tokens["input_ids"],
            attention_mask=input_tokens["attention_mask"],
        )

        # 1. Extract logits for the last token position
        logits = outputs.logits[0, -1, :]
        prob_over_tokens = F.softmax(logits, dim=-1)

        # 2. Sort probabilities in descending order
        sorted_probs, sorted_indices = torch.sort(prob_over_tokens, descending=True)

        # 3. Calculate cumulative sum
        cum_sum_probs = torch.cumsum(sorted_probs, dim=-1)

        # 4. Correctly find the cutoff index
        # We find the first index where cum_sum exceeds the threshold
        # Using .item() converts it to a standard Python integer safely
        cutoff_mask = cum_sum_probs >= thresh
        if not cutoff_mask.any():
            cutoff_index = len(sorted_probs)
        else:
            cutoff_index = int(torch.argmax(cutoff_mask.byte()).item()) + 1

        print(f"Choosing from {cutoff_index} tokens")

        # 5. Keep only the nucleus tokens, zero out the rest
        nucleus_probs = torch.zeros_like(sorted_probs)
        nucleus_probs[:cutoff_index] = sorted_probs[:cutoff_index]

        # 6. Renormalize the remaining probabilities
        nucleus_probs = nucleus_probs / torch.sum(nucleus_probs)

        # 7. Sample from the filtered distribution
        sampled_sorted_idx = torch.multinomial(nucleus_probs, num_samples=1)

        # Map back to the original token ID space
        next_token_id = sorted_indices[sampled_sorted_idx].unsqueeze(0)  # Shape: (1, 1)

        # 8. Append token to inputs (maintaining correct device placement)
        device = input_tokens["input_ids"].device
        output_tokens = input_tokens.copy()

        output_tokens["input_ids"] = torch.cat(
            (output_tokens["input_ids"], next_token_id), dim=1
        )
        output_tokens["attention_mask"] = torch.cat(
            (
                output_tokens["attention_mask"],
                torch.ones((1, 1), dtype=torch.long, device=device),
            ),
            dim=1,
        )
        output_tokens["last_token_prob"] = prob_over_tokens[next_token_id.item()].item()

        return output_tokens

In [51]:
# Expected output:
# The best thing about Bath is that it's not a city that has been around
set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors="pt")
for i in range(10):
    input_tokens = get_nucleus_sampling_token(
        input_tokens, model, tokenizer, thresh=0.2
    )
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

Choosing from 1 tokens
The best thing about Bath is that
Choosing from 1 tokens
The best thing about Bath is that it
Choosing from 1 tokens
The best thing about Bath is that it's
Choosing from 3 tokens
The best thing about Bath is that it's not
Choosing from 2 tokens
The best thing about Bath is that it's not a
Choosing from 26 tokens
The best thing about Bath is that it's not a city
Choosing from 3 tokens
The best thing about Bath is that it's not a city that
Choosing from 2 tokens
The best thing about Bath is that it's not a city that has
Choosing from 2 tokens
The best thing about Bath is that it's not a city that has been
Choosing from 12 tokens
The best thing about Bath is that it's not a city that has been around


In [52]:
# Expected output:
# The best thing about Bath is that it's not a city that has been around
set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors="pt")
for i in range(10):
    input_tokens = get_nucleus_sampling_token_torch(
        input_tokens, model, tokenizer, thresh=0.2
    )
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

Choosing from 1 tokens
The best thing about Bath is that
Choosing from 1 tokens
The best thing about Bath is that it
Choosing from 1 tokens
The best thing about Bath is that it's
Choosing from 3 tokens
The best thing about Bath is that it's a
Choosing from 7 tokens
The best thing about Bath is that it's a place
Choosing from 1 tokens
The best thing about Bath is that it's a place where
Choosing from 1 tokens
The best thing about Bath is that it's a place where you
Choosing from 1 tokens
The best thing about Bath is that it's a place where you can
Choosing from 4 tokens
The best thing about Bath is that it's a place where you can go
Choosing from 1 tokens
The best thing about Bath is that it's a place where you can go to


In [53]:
# Expected output:
# The best thing about Bath is that it's not a city that has been around
set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors="pt")
for i in range(10):
    input_tokens = get_nucleus_sampling_token(
        input_tokens, model, tokenizer, thresh=0.2
    )
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

Choosing from 1 tokens
The best thing about Bath is that
Choosing from 1 tokens
The best thing about Bath is that it
Choosing from 1 tokens
The best thing about Bath is that it's
Choosing from 3 tokens
The best thing about Bath is that it's not
Choosing from 2 tokens
The best thing about Bath is that it's not a
Choosing from 26 tokens
The best thing about Bath is that it's not a city
Choosing from 3 tokens
The best thing about Bath is that it's not a city that
Choosing from 2 tokens
The best thing about Bath is that it's not a city that has
Choosing from 2 tokens
The best thing about Bath is that it's not a city that has been
Choosing from 12 tokens
The best thing about Bath is that it's not a city that has been around


In [54]:
# TODO -- experiment with setting the threshold probability to larger or smaller values
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors="pt")
for i in range(10):
    input_tokens = get_nucleus_sampling_token(
        input_tokens, model, tokenizer, thresh=0.2
    )
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

Choosing from 1 tokens
The best thing about Bath is that
Choosing from 1 tokens
The best thing about Bath is that it
Choosing from 1 tokens
The best thing about Bath is that it's
Choosing from 3 tokens
The best thing about Bath is that it's so
Choosing from 4 tokens
The best thing about Bath is that it's so much
Choosing from 1 tokens
The best thing about Bath is that it's so much more
Choosing from 1 tokens
The best thing about Bath is that it's so much more than
Choosing from 1 tokens
The best thing about Bath is that it's so much more than just
Choosing from 1 tokens
The best thing about Bath is that it's so much more than just a
Choosing from 5 tokens
The best thing about Bath is that it's so much more than just a beach


# Beam search

All of the methods we've seen so far choose the tokens one by one.  But this isn't necessarily sensible.  Even greedily choosing the best token doesn't necessarily retrieve the sequence with the highest probability.  It might be that the most likely token only has very unlikely tokens following it.

Beam search maintains $K$ hypotheses about the best possible continuation.  It starts with the top $K$ continuations.  Then for each of those, it finds the top K continuations, giving $K^2$ hypotheses.  Then it retains just the top $K$ of these so that the number of hypotheses stays the same.

In [83]:
device = "cpu"

In [91]:
# This routine returns the k'th most likely next token.
# If k =0 then it returns the most likely token, if k=1 it returns the next most likely and so on
# We will need this for beam search
def get_kth_most_likely_token(input_tokens, model, tokenizer, k):
    # Run model to get prediction over next output
    outputs = model(
        input_ids=input_tokens["input_ids"],
        attention_mask=input_tokens["attention_mask"],
    )
    # Find prediction
    prob_over_tokens = F.softmax(outputs.logits, dim=-1).cpu().detach().numpy()[0, -1]

    # Find the k'th most likely token
    # TODO Sort the probabilities from largest to smallest
    # Replace this line:
    sorted_prob_over_tokens = np.sort(prob_over_tokens)[::-1]
    # TODO Find the k'th sorted probability
    # Replace this line
    kth_prob_value = sorted_prob_over_tokens[k]

    # Find position of this token.
    next_token = np.where(prob_over_tokens == kth_prob_value)[0]

    # Append token to sentence
    output_tokens = input_tokens
    output_tokens["input_ids"] = torch.cat(
        (output_tokens["input_ids"], torch.tensor([next_token], device=device)), dim=1
    )
    output_tokens["attention_mask"] = torch.cat(
        (output_tokens["attention_mask"], torch.tensor([[1]], device=device)), dim=1
    )
    output_tokens["last_token_prob"] = prob_over_tokens[next_token]
    output_tokens["log_prob"] = output_tokens["log_prob"] + np.log(
        prob_over_tokens[next_token]
    )
    return output_tokens

In [92]:
device = "cpu"

In [93]:
model = model.to(device)

In [94]:
# We can test this code and see that if we choose the 2nd most likely (K=1) token each time
# then we get much better generation results than if we choose the 2001st most likely token

# Expected output:
# The best thing about Bath is the way you get the most bang outta the
set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors="pt").to(device)
input_tokens["log_prob"] = 0.0
for i in range(10):
    input_tokens = get_kth_most_likely_token(input_tokens, model, tokenizer, k=1)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

# Expected output:
# The best thing about Bath is mixed profits partnerships» buy generic+ Honda throttlecont
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors="pt").to(device)
input_tokens["log_prob"] = 0.0
for i in range(10):
    input_tokens = get_kth_most_likely_token(input_tokens, model, tokenizer, k=2000)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

# TODO -- play around with different values of K

The best thing about Bath is the
The best thing about Bath is the way
The best thing about Bath is the way you
The best thing about Bath is the way you get
The best thing about Bath is the way you get the
The best thing about Bath is the way you get the most
The best thing about Bath is the way you get the most bang
The best thing about Bath is the way you get the most bang out
The best thing about Bath is the way you get the most bang outta
The best thing about Bath is the way you get the most bang outta the
The best thing about Bath is mixed
The best thing about Bath is mixed profits
The best thing about Bath is mixed profits partnerships
The best thing about Bath is mixed profits partnerships»
The best thing about Bath is mixed profits partnerships» buy
The best thing about Bath is mixed profits partnerships» buy generic
The best thing about Bath is mixed profits partnerships» buy generic+
The best thing about Bath is mixed profits partnerships» buy generic+drive
The best thing abou

In [95]:
model = model.to("cpu")

In [105]:
tokenizer.decode(input_tokens["input_ids"])

['The best thing about Bath is']

In [100]:
# Print out each beam plus the log probability
def print_beams(beams):
    for index, beam in enumerate(beams):
        print("Beam %d, Prob %3.3f: " % (index, beam["log_prob"]))
        # + tokenizer.decode(beam["input_ids"][0], skip_special_tokens=True)

    print("---")


# TODO:  Read this code carefully!
def do_beam_search(input_tokens_in, model, tokenizer, n_beam=5, beam_length=10):
    # Store beams in a list
    input_tokens["log_prob"] = 0.0

    # Initialize with n_beam most likely continuations
    beams = [None] * n_beam
    for c_k in range(n_beam):
        beams[c_k] = dict(input_tokens_in)
        beams[c_k] = get_kth_most_likely_token(beams[c_k], model, tokenizer, c_k)

    print_beams(beams)

    # For each token in the sequence we will add
    for c_pos in range(beam_length - 1):
        # Now for each beam, we continue it in the most likely ways, making n_beam*n_beam type hypotheses
        beams_all = [None] * (n_beam * n_beam)
        log_probs_all = np.zeros(n_beam * n_beam)
        # For each current hypothesis
        for c_beam in range(n_beam):
            # For each continuation
            for c_k in range(n_beam):
                # Store the continuation and the probability
                beams_all[c_beam * n_beam + c_k] = dict(
                    get_kth_most_likely_token(beams[c_beam], model, tokenizer, c_k)
                )
                log_probs_all[c_beam * n_beam + c_k] = beams_all[c_beam * n_beam + c_k][
                    "log_prob"
                ]

        # Keep the best n_beams sequences with the highest probabilities
        sorted_index = np.argsort(np.array(log_probs_all) * -1)
        for c_k in range(n_beam):
            beams[c_k] = dict(beams_all[sorted_index[c_k]])

        # Print the beams
        print_beams(beams)

    return beams[0]

In [ ]:
# Expected output:
# The best thing about Bath is that it's a place where you don't have to

set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors="pt")

# Now let's call the beam search
# It takes a while as it has to run the model multiple times to add a token
n_beams = 5
best_beam = do_beam_search(input_tokens, model, tokenizer)
print("Beam search result:")
print(tokenizer.decode(best_beam["input_ids"][0], skip_special_tokens=True))

# You should see that the best answer is not the same as the greedy solution we found above

In [98]:
print(input_tokens)

{'input_ids': tensor([[  464,  1266,  1517,   546, 24967,   318]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]]), 'log_prob': 0.0}


You can read about more decoding strategies in this blog (which uses a recursive neural network, not a transformer, but the principles are the same).

https://www.borealisai.com/research-blogs/tutorial-6-neural-natural-language-generation-decoding-algorithms/

You can also look at other possible language models via hugging face:

https://huggingface.co/docs/transformers/v4.25.1/en/model_summary#decoders-or-autoregressive-models
